[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C06_Interpretability_Course/03_attention_induction/03_induction_heads.ipynb)

# 03 · QK/OV 电路与 Induction Heads：手工构造会 in-context copy 的 transformer

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span> 纯 numpy + matplotlib，无 torch、无下载、秒级运行。

**本 notebook 你将完成：**

1. 实现**单头 causal attention** 前向（one-hot embedding + 分块 residual stream，电路肉眼可读）；
2. **手工构造 Layer 1 previous-token head**（QK 只看 position embedding，使位置 $i$ 精确注意 $i-1$）并可视化 attention pattern；
3. **手工构造 Layer 2 induction head**（K-composition：query 读当前 token，key 读 Layer 1 写入的"前一个 token"），在随机重复序列上验证**第二遍出现时 top-1 预测全对**；
4. 定义并计算 **induction score** 与 **copy score**；
5. 验证 **OOD 泛化**：同一套手写权重对 200 条全新随机序列全部成功——induction 电路是**算法**，不是记忆；
6. 4 道 ✏️ 练习巩固 QK/OV 电路分析。

参考：[Elhage 2021] *A Mathematical Framework for Transformer Circuits* (transformer-circuits.pub)、[Olsson 2022] *In-context Learning and Induction Heads* (arXiv:2209.11895)、[Jain & Wallace 2019] *Attention is not Explanation* (arXiv:1902.10186)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# ---- 模型配置：one-hot embedding，residual stream 按"块"划分，电路直接可读 ----
V    = 8       # 词表大小（token 0 保留作 BOS，pattern token 取 1..7）
P    = 24      # 最大序列长度（position embedding 个数）
BETA = 16.0    # QK 打分放大系数：softmax 近似 one-hot（硬注意力）

# residual stream 维度布局: [ token | position | PREV 暂存 | OUT 输出 ]
d    = V + P + V + V
TOK  = slice(0,         V)          # 当前 token one-hot（embedding 写入）
POS  = slice(V,         V + P)      # 位置 one-hot（embedding 写入）
PREV = slice(V + P,     V + P + V)  # Layer 1 写入: "我的前一个 token 是谁"
OUT  = slice(V + P + V, d)          # Layer 2 写入: 预测的 token（W_U 直接读它当 logits）

E_tok = np.zeros((V, d)); E_tok[np.arange(V), np.arange(V)]             = 1.0  # token t -> TOK 块
E_pos = np.zeros((P, d)); E_pos[np.arange(P), V + np.arange(P)]         = 1.0  # pos  p -> POS 块
W_U   = np.zeros((V, d)); W_U[np.arange(V), V + P + V + np.arange(V)]   = 1.0  # unembedding: 读 OUT 块

def embed(tokens):
    """tokens (n,) int -> X (n, d)。x_i = token one-hot + position one-hot。"""
    tokens = np.asarray(tokens)
    return E_tok[tokens] + E_pos[np.arange(len(tokens))]

def softmax(s, axis=-1):
    s = s - s.max(axis=axis, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=axis, keepdims=True)

def attn_head(X, Wq, Wk, Wv, Wo):
    """单头 causal attention。返回 (写回 residual stream 的输出, attention pattern)。
    打分不除 sqrt(d_head)——缩放已吸收进 BETA。"""
    n = X.shape[0]
    Q, K, Vv = X @ Wq.T, X @ Wk.T, X @ Wv.T
    scores = Q @ K.T
    scores[np.triu_indices(n, k=1)] = -np.inf       # causal mask: 看不到 j > i
    pattern = softmax(scores, axis=-1)
    return (pattern @ Vv) @ Wo.T, pattern

print(f"d_model = {d}  (TOK {V} + POS {P} + PREV {V} + OUT {V})")

## 1 · Layer 1：previous-token head（QK 只看位置）

一个 attention head 由两条独立电路决定（[Elhage 2021]）：**QK 电路** $W_Q^\top W_K$ 决定"看哪里"，**OV 电路** $W_O W_V$ 决定"搬什么"。

previous-token head 的 QK 电路与 token 内容完全无关、只看位置：

$$ q_i = \sqrt{\beta}\, e_{\mathrm{pos}=i}, \qquad k_j = \sqrt{\beta}\, e_{\mathrm{pos}=j+1} \;\Longrightarrow\; q_i \cdot k_j = \beta \cdot \mathbf{1}[\,j = i-1\,] $$

即 $W_K$ 把位置 $p$ 的 key 放进"槽位 $p+1$"，谁的 query 落在这个槽位？正是位置 $p+1$。$\beta$ 足够大时 softmax 近似 one-hot：**位置 $i$ 几乎只注意 $i-1$**。

OV 电路则把被注意位置的 token one-hot 原样搬运，写进 residual stream 的 **PREV 块**——相当于在每个位置贴一张便签："我的前一个 token 是 X"。这张便签就是第二层 K-composition 的原料。

（边界：位置 0 没有"前一个"，所有打分为 0，softmax 退化为只注意自己，把自己的 token（BOS）写进 PREV——这正是我们保留 BOS 的原因，见第 2 节。）

In [ ]:
# ---- Layer 1: previous-token head ----
sb  = np.sqrt(BETA)
Wq1 = np.zeros((P, d)); Wq1[:, POS] = sb * np.eye(P)        # q_i = √β · onehot(pos i)
Wk1 = np.zeros((P, d))
Wk1[1:, V:V + P - 1] = sb * np.eye(P - 1)                   # 位置 p 的 key 落在"槽位 p+1"

Wv1 = np.zeros((V, d)); Wv1[:, TOK]  = np.eye(V)            # value = 被注意位置的 token one-hot
Wo1 = np.zeros((d, V)); Wo1[PREV, :] = np.eye(V)            # 写进 PREV 块

tokens_demo = [0, 3, 1, 6, 2, 5, 3, 1, 6, 2, 5]             # 0 是 BOS
X0 = embed(tokens_demo)
h1, pat1 = attn_head(X0, Wq1, Wk1, Wv1, Wo1)
X1 = X0 + h1                                                # 残差连接

n = len(tokens_demo)
assert np.all(pat1[np.arange(1, n), np.arange(0, n - 1)] > 0.99), "位置 i 应注意 i-1"
for i in range(1, n):
    assert X1[i, PREV].argmax() == tokens_demo[i - 1], "PREV 块应写入前一个 token"

fig, ax = plt.subplots(figsize=(4.6, 4.2))
im = ax.imshow(pat1, cmap="Blues", vmin=0, vmax=1)
ax.set_title("Layer 1: previous-token head")
ax.set_xlabel("key 位置 j"); ax.set_ylabel("query 位置 i")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()
print("✅ pattern 集中在次对角线 j = i-1；PREV 块 = 前一个 token 的 one-hot")

## 2 · Layer 2：induction head（K-composition）

induction 算法："序列里出现过 $\dots A\;B \dots$，当前 token 又是 $A$ ⟹ 预测 $B$"。第二层这样实现：

$$ q_i = \sqrt{\beta}\,\cdot\,\underbrace{\text{onehot}(t_i)}_{\text{读 TOK 块（embedding 原始信息）}}, \qquad k_j = \sqrt{\beta}\,\cdot\,\underbrace{\text{onehot}(t_{j-1})}_{\text{读 PREV 块（Layer 1 的输出！）}} $$

打分 $q_i \cdot k_j = \beta\cdot\mathbf{1}[\,t_i = t_{j-1}\,]$："你的前一个 token 等于我现在的 token" ⟹ 位置 $i$ 注意**上一次 $t_i$ 出现位置的下一格**。key 一侧用到了上一层 head 的输出，这就是 **K-composition**——单层模型做不到这件事（query/key 都只能来自 embedding，只能查 skip-trigram 表），两层的质变全在这里。

OV 电路仍是纯 copy：把被注意位置的 token 抄进 **OUT 块**，$W_U$ 读 OUT 当 logits。full OV 电路 $W_U W_O W_V W_E = I_{|V|}$——"注意谁，就预测谁"。

为什么要 BOS？位置 0 的 PREV 写的是它自己的 token。若它是普通 token $X$，则查询 $t_i = X$ 时位置 0 与正确目标会**平分注意力**、搬运被污染。让 token 0 = BOS 且从不出现在 pattern 里，位置 0 的 key 永远不被匹配。验证重复序列 $[\text{BOS}, A B C D E F, A B C D E F]$（周期 $T$）：induction 命中的应是 $j = i - T + 1$。

In [ ]:
# ---- Layer 2: induction head ----
Wq2 = np.zeros((V, d)); Wq2[:, TOK]  = sb * np.eye(V)   # query 读当前 token
Wk2 = np.zeros((V, d)); Wk2[:, PREV] = sb * np.eye(V)   # key 读 PREV 块 <- K-composition!
Wv2 = np.zeros((V, d)); Wv2[:, TOK]  = np.eye(V)        # value = 被注意位置的 token
Wo2 = np.zeros((d, V)); Wo2[OUT, :]  = np.eye(V)        # 写进 OUT 块（logits）

def run_model(tokens):
    """两层 attention-only transformer 前向。返回 (logits, pat1, pat2)。"""
    X0 = embed(tokens)
    h1, p1 = attn_head(X0, Wq1, Wk1, Wv1, Wo1)
    X1 = X0 + h1
    h2, p2 = attn_head(X1, Wq2, Wk2, Wv2, Wo2)
    X2 = X1 + h2
    return X2 @ W_U.T, p1, p2

# 随机重复序列: [BOS, A B C D E F, A B C D E F]（pattern 内 token 互不相同）
period = 6
pattern_toks = rng.permutation(np.arange(1, V))[:period]
seq = np.concatenate([[0], pattern_toks, pattern_toks])
print("序列:", seq, f"  (周期 T = {period}, 位置 0 是 BOS)")

logits, pat1_r, pat2_r = run_model(seq)

fig, ax = plt.subplots(figsize=(4.6, 4.2))
im = ax.imshow(pat2_r, cmap="Oranges", vmin=0, vmax=1)
ax.set_title("Layer 2: induction head（第二遍出现条纹 j = i−T+1）")
ax.set_xlabel("key 位置 j"); ax.set_ylabel("query 位置 i")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 3 · 验证 in-context copy + 两个操作化指标

逐位置看 top-1 预测。**第一遍**（pattern 第一次出现）：没有任何 key 匹配，layer-2 注意力退化为均匀分布，OUT 块是前缀 token 的平均——预测毫无信息。**第二遍**：每个位置恰有一个 key 匹配（"上一次出现的下一格"），把正确的下一个 token 以 $\approx 1$ 的概率搬进 logits。

再把"是不是 induction head"操作化为两个机械指标（[Olsson 2022] 的 prefix-matching / copying score 的简化版）：

- **induction score**：重复序列（周期 $T$）上，attention pattern 在对角线 $(i,\; i-T+1)$ 上的平均质量。手工 head 应 $\approx 1$，均匀注意基线 $\approx \frac{1}{i+1}$；
- **copy score**：full OV 电路 $W_U W_O W_V W_E \in \mathbb{R}^{|V|\times|V|}$ 的对角占优程度——"注意到 token $t$ 是否提升 $t$ 自己的 logit"。这是**纯权重性质**，不用跑前向。

In [ ]:
L    = len(seq)
pred = logits.argmax(axis=-1)

print(f"{'pos':>3} {'token':>5} {'预测':>4} {'真值':>4}   phase")
for i in range(L - 1):
    phase = "第一遍" if i <= period else "第二遍"
    mark  = "✓" if pred[i] == seq[i + 1] else "✗"
    print(f"{i:>3} {seq[i]:>5} {pred[i]:>4} {seq[i + 1]:>4}   {phase} {mark}")

first  = [pred[i] == seq[i + 1] for i in range(1, period)]           # 第一遍可评位置
second = [pred[i] == seq[i + 1] for i in range(period + 1, L - 1)]   # 第二遍可评位置
acc1, acc2 = float(np.mean(first)), float(np.mean(second))
print(f"\nper-token top-1 正确率: 第一遍 = {acc1:.2f}   第二遍 = {acc2:.2f}")
assert acc2 == 1.0, "induction 电路应使第二遍全对"
assert acc1 < 0.5,  "第一遍不应有信息"

# induction score: pattern 在 (i, i-T+1) 对角线上的平均
idx       = np.arange(period + 1, len(seq))
ind_score = pat2_r[idx, idx - period + 1].mean()
baseline  = np.mean(1.0 / (idx + 1))                  # 均匀 causal 注意的基线

# copy score: full OV 电路的每一列 argmax 是否都落在对角线上
full_ov   = W_U @ Wo2 @ Wv2 @ E_tok.T                 # (V, V): 列 = 被注意 token, 行 = 输出 logit
copy_sc   = np.mean(full_ov.argmax(axis=0) == np.arange(V))

print(f"induction score = {ind_score:.4f}   (均匀注意基线 ≈ {baseline:.3f})")
print(f"copy score      = {copy_sc:.2f}")
print("\nfull OV 电路矩阵 W_U·W_O·W_V·W_E（应为单位阵 = 完美 copy）:")
print(full_ov.round(2))
assert ind_score > 0.99 and copy_sc == 1.0

## 4 · OOD 泛化：电路是算法，不是记忆

这套权重从未"训练"，更没见过任何特定序列。把**周期、token 全部随机重抽** 200 次——如果 induction 行为是某种记忆，换序列就会失效；如果是算法，应每条都成功。这正是电路分析的价值主张：**理解了机制，就能预测它在分布外的行为**（对比：行为评测只能告诉你它在测过的分布上表现如何）。

In [ ]:
accs, ind_scores = [], []
for trial in range(200):
    T  = int(rng.integers(3, 8))                              # 周期 3..7 随机
    pt = rng.permutation(np.arange(1, V))[:T]                 # token 随机重抽
    s  = np.concatenate([[0], pt, pt])
    lg, _, p2 = run_model(s)
    pr = lg.argmax(axis=-1)
    accs.append(np.mean([pr[i] == s[i + 1] for i in range(T + 1, len(s) - 1)]))
    ii = np.arange(T + 1, len(s))
    ind_scores.append(p2[ii, ii - T + 1].mean())

print(f"200 条全新随机重复序列: 第二遍正确率 min = {min(accs):.3f}, mean = {np.mean(accs):.3f}")
print(f"induction score:        min = {min(ind_scores):.4f}")
assert min(accs) == 1.0, "同一套权重应对任意随机序列都成功"

plt.figure(figsize=(5.2, 3))
plt.hist(ind_scores, bins=20, color="#f0a868")
plt.xlabel("induction score"); plt.ylabel("# 序列")
plt.title("同一套手写权重 × 200 条全新序列")
plt.tight_layout(); plt.show()

---
## ✏️ 练习 1：实现 `prev_token_score` 与 `induction_score`

把上文的两个 pattern 指标封装成通用函数：

- `prev_token_score(pattern)`：对 $i = 1, \dots, n-1$ 取 `pattern[i, i-1]` 的平均；
- `induction_score(pattern, period)`：对 $i = \mathrm{period}+1, \dots, n-1$ 取 `pattern[i, i-period+1]` 的平均。

**提示**：`np.arange` 构造行/列索引，各 2–3 行。注意起始下标——若从 $i=0$（或 $i \le \mathrm{period}$）开始，列索引为负，numpy **不报错而是回卷到行尾**，score 会被悄悄污染，这是 pattern 分析代码里最常见的 bug。

In [ ]:
def prev_token_score(pattern):
    # TODO: 对 i = 1..n-1 取 pattern[i, i-1] 的平均
    raise NotImplementedError

def induction_score(pattern, period):
    # TODO: 对 i = period+1..n-1 取 pattern[i, i-period+1] 的平均
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
n_r = len(seq)
uniform = np.tril(np.ones((n_r, n_r))) / np.arange(1, n_r + 1)[:, None]  # 均匀 causal 基线
assert prev_token_score(pat1_r) > 0.99,            "手工 previous-token head 应接近 1"
assert induction_score(pat2_r, period) > 0.99,     "手工 induction head 应接近 1"
assert prev_token_score(uniform) < 0.5,            "均匀注意基线应远小于 1"
assert induction_score(uniform, period) < 0.5
assert abs(prev_token_score(np.eye(5))) < 1e-12,   "纯自注意 pattern 的 prev-token score = 0"
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 full QK 电路矩阵

attention 打分 $s_{ij} = q_i \cdot k_j = x_i^\top W_Q^\top W_K\, x_j$。把输入限定在两组 one-hot 基上，就得到**端到端 QK 表**：

`full_qk(E_left, Wq, Wk, E_right)` 返回矩阵 $M = E_{\text{left}}\, W_Q^\top W_K\, E_{\text{right}}^\top$，其中 `E_left (n_q, d)`、`E_right (n_k, d)` 的每一行是一个基向量；$M[a, b]$ = "query 来自基 $a$、key 来自基 $b$ 时的打分"。

验证手工构造的结构（这正是讲解页第 2 节的 $W_E^\top W_Q^\top W_K W_E$）：

- Layer 1 在 position 基下应是 $\beta \cdot$ **下移一格的 shift 矩阵**（`np.eye(P, k=-1)`）；
- Layer 2 在（token 基，PREV 基）下应是 $\beta \cdot I_V$。

**提示**：一行矩阵乘法。方向别搞反：**行是 query、列是 key**；`Wq.T @ Wk` 而不是 `Wq @ Wk.T`（两个 `W` 都是 `(d_head, d)`）。

In [ ]:
def full_qk(E_left, Wq, Wk, E_right):
    # TODO: 返回 E_left @ Wq.T @ Wk @ E_right.T   （行 = query 基, 列 = key 基）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
E_prev = np.zeros((V, d)); E_prev[np.arange(V), V + P + np.arange(V)] = 1.0  # token t -> PREV 块基向量
M1 = full_qk(E_pos, Wq1, Wk1, E_pos)      # (P, P): 位置 × 位置
M2 = full_qk(E_tok, Wq2, Wk2, E_prev)     # (V, V): 当前 token × "前一个 token"
assert M1.shape == (P, P) and M2.shape == (V, V)
assert np.allclose(M1, BETA * np.eye(P, k=-1)), "Layer1 QK 应为 β·shift 矩阵（i 注意 i-1）"
assert np.allclose(M2, BETA * np.eye(V)),       "Layer2 QK 在 (token, PREV) 基下应为 β·单位阵"
print("✅ 练习 2 通过")

## ✏️ 练习 3：repeated-sequence loss 对比——第一遍 vs 第二遍

把第 3 节的对比封装成函数 `pass_accuracies(pattern_toks)`：给定一段互不相同的 pattern token（取值 1..V-1），构造序列 `[BOS] + pattern + pattern`，跑 `run_model`，返回 `(acc_first, acc_second)`：

- `acc_first`：位置 $1 \dots T-1$ 上 `pred[i] == seq[i+1]` 的比例（$T$ = pattern 长度）；
- `acc_second`：位置 $T+1 \dots 2T-1$ 上的比例。

这是 [Olsson 2022] 用来度量 ICL 的 per-token loss 对比的 argmax 简化版。

**提示**：复用 `run_model` 与 `np.concatenate`，约 8 行。注意：位置 $i$ 预测的是 `seq[i+1]`，所以最后一个位置（$2T$）没有真值可评；位置 $T$ 跨越两遍边界（当前 token 是第一次出现的 F），两边都不算。

In [ ]:
def pass_accuracies(pattern_toks):
    # TODO:
    #  1) seq_ = [0] + pattern + pattern   （0 是 BOS）
    #  2) logits, _, _ = run_model(seq_); pred = argmax
    #  3) acc_first  = 位置 1..T-1   上 pred[i] == seq_[i+1] 的比例
    #  4) acc_second = 位置 T+1..2T-1 上 pred[i] == seq_[i+1] 的比例
    #  返回 (acc_first, acc_second)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
a1, a2 = pass_accuracies(np.array([3, 1, 6, 2, 5]))
assert a2 == 1.0, "第二遍应全对"
assert a1 < 0.5,  "第一遍不应有信息"
rng3 = np.random.default_rng(0)
results = [pass_accuracies(rng3.permutation(np.arange(1, V))[:4]) for _ in range(20)]
assert all(r[1] == 1.0 for r in results), "任意随机 pattern 第二遍都应全对"
assert np.mean([r[0] for r in results]) < 0.5
print("✅ 练习 3 通过")

## ✏️ 练习 4：用 OV 电路验证"搬运的就是被注意的 token"

QK 决定看哪里，OV 决定搬什么——两条电路独立，要分别验证。实现两个函数：

- `ov_circuit(Wu, Wo, Wv, E)`：返回 full OV 矩阵 $M = W_U W_O W_V E^\top \in \mathbb{R}^{V\times V}$。$M[:, t]$ = "注意到 token $t$ 时对各输出 logit 的贡献"；
- `copy_score(M)`：每一列的 argmax 落在对角线上的比例（=1 说明"注意谁就预测谁"，是纯 copy 电路）。

自测还会做一个**行为级交叉验证**：在重复序列的第二遍上，逐位置检查 `logits 的 argmax == 被注意位置（pattern argmax）的 token`——权重层面的结论与前向行为应互相印证。

**提示**：各 1–2 行。`E` 形状 `(V, d)`（每行一个 token 基向量），所以最后乘 `E.T`；`copy_score` 用 `M.argmax(axis=0) == np.arange(...)` 取均值。

In [ ]:
def ov_circuit(Wu, Wo, Wv, E):
    # TODO: 返回 Wu @ Wo @ Wv @ E.T   （(V_out, V_in) 的 full OV 矩阵）
    raise NotImplementedError

def copy_score(M):
    # TODO: 返回每列 argmax 落在对角线上的比例
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
M = ov_circuit(W_U, Wo2, Wv2, E_tok)
assert M.shape == (V, V)
assert np.allclose(M, np.eye(V)), "手工 induction head 的 full OV 应是单位阵"
assert copy_score(M) == 1.0
# 行为级交叉验证：第二遍每个位置，预测的 = 被注意位置的 token
lg, _, p2 = run_model(seq)
for i in range(period + 1, len(seq)):
    src = p2[i].argmax()
    assert lg[i].argmax() == seq[src], "OV 搬运的应是被注意位置的 token"
# 随机 W_O 不是 copy 电路
rng4   = np.random.default_rng(0)
M_rand = ov_circuit(W_U, rng4.normal(size=(d, V)) / np.sqrt(d), Wv2, E_tok)
assert copy_score(M_rand) < 0.6
print("✅ 练习 4 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def prev_token_score(pattern):
    i = np.arange(1, pattern.shape[0])
    return pattern[i, i - 1].mean()

def induction_score(pattern, period):
    i = np.arange(period + 1, pattern.shape[0])
    return pattern[i, i - period + 1].mean()

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def full_qk(E_left, Wq, Wk, E_right):
    return E_left @ Wq.T @ Wk @ E_right.T

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
def pass_accuracies(pattern_toks):
    T    = len(pattern_toks)
    seq_ = np.concatenate([[0], pattern_toks, pattern_toks])
    logits_, _, _ = run_model(seq_)
    pred_ = logits_.argmax(axis=-1)
    acc_first  = float(np.mean([pred_[i] == seq_[i + 1] for i in range(1, T)]))
    acc_second = float(np.mean([pred_[i] == seq_[i + 1] for i in range(T + 1, 2 * T)]))
    return acc_first, acc_second

In [ ]:
# 练习 4 参考答案（先自己做，再对照）
def ov_circuit(Wu, Wo, Wv, E):
    return Wu @ Wo @ Wv @ E.T

def copy_score(M):
    return float(np.mean(M.argmax(axis=0) == np.arange(M.shape[0])))

---
## 小结

- attention head = **QK 电路**（$W_Q^\top W_K$，决定看哪里）× **OV 电路**（$W_O W_V$，决定搬什么），两条电路独立、可分别逆向；one-hot embedding 下 full QK / full OV 是肉眼可读的 $|V|\times|V|$ 表。
- **previous-token head**（纯位置 QK + copy OV 写 PREV 块）+ **K-composition**（第二层 key 读 PREV 块）= **induction head**：找到当前 token 上次出现的下一格、抄过来。1 层模型只能查 skip-trigram 表，做不到。
- 操作化指标：**induction score**（pattern 在 $(i, i-T+1)$ 对角线的质量）与 **copy score**（full OV 对角占优）。
- 同一套手写权重对任意随机序列 100% 成功——**电路是算法而非记忆**，这是机制理解支持 OOD 行为预测的最小完整示例。
- 但注意（讲解页第 6 节）：真实模型里 attention pattern ≠ 解释 [Jain & Wallace 2019]，声称功能必须配因果检验——下一模块 **04 · Activation Patching** 就补上这块。

---
## 🎯 真实数据胶囊题：真实文本上的 induction（看到 [A][B]…[A] 就预测 [B]）

induction head 实现 in-context 复制：之前出现过 A 后跟 B，再见到 A 就预测 B。用真实莎士比亚文本(字符级)，实现 induction 规则预测下一个字符，验证它远好于随机。

> 本模块新增的**真实数据**练习：用**真实 GPT-2 权重/embedding**把本章的可解释性技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, struct, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.interp_data"); os.makedirs(CACHE,exist_ok=True)
ST="https://huggingface.co/openai-community/gpt2/resolve/main/model.safetensors"
def _rng(s,e):
    req=urllib.request.Request(ST, headers={"Range":f"bytes={s}-{e}"})
    return urllib.request.urlopen(req,timeout=60).read()
def gpt2_emb_block(n=6000):
    cache=os.path.join(CACHE,f"wte_{n}.npy")
    if os.path.exists(cache): return np.load(cache)
    hlen=struct.unpack("<Q", _rng(0,7))[0]; hdr=json.loads(_rng(8,8+hlen-1))
    info=hdr["wte.weight"]; base=8+hlen; s0=info["data_offsets"][0]; d=info["shape"][1]
    raw=_rng(base+s0, base+s0+n*d*4-1)
    E=np.frombuffer(raw,dtype=np.float32).reshape(n,d).copy()
    np.save(cache,E); return E
def gpt2_vocab():
    p=os.path.join(CACHE,"vocab.json")
    if not os.path.exists(p): urllib.request.urlretrieve("https://huggingface.co/openai-community/gpt2/resolve/main/vocab.json",p)
    return json.load(open(p))
def digit_letter_dataset(lim=6000):
    "返回 (X[token嵌入], y[1=数字 0=字母], E, ids_digit, ids_alpha)"
    v=gpt2_vocab(); E=gpt2_emb_block(lim)
    dig=[i for t,i in v.items() if i<lim and t.isdigit()]
    alpha=[i for t,i in v.items() if i<lim and t.isalpha() and t.isascii()]
    rng=np.random.default_rng(0); alpha=list(rng.permutation(alpha)[:len(dig)])
    ids=dig+alpha; y=np.array([1]*len(dig)+[0]*len(alpha))
    return E[ids], y, E, dig, alpha
def shakespeare():
    p=os.path.join(CACHE,"shake.txt")
    if not os.path.exists(p): urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",p)
    return open(p).read()

txt=shakespeare()[:30000]
print(f"真实文本 {len(txt)} 字符, 唯一字符 {len(set(txt))}")

**练习**：实现 `induction_predict(text, i)`：在位置 i，找**之前**最近一次出现 `text[i]` 的位置 j，预测下一个字符为 `text[j+1]`（没有先例则返回 None）。统计它的下一字符预测准确率。

In [ ]:
def induction_predict(text, i):
    # TODO: 在 text[:i] 里找最后一个等于 text[i] 的位置 j(且 j+1<i)，返回 text[j+1]；无则 None
    raise NotImplementedError


In [ ]:
# 自测：induction 准确率应远高于随机(1/字符数)
hit=tot=0
for i in range(2, 5000):
    pred=induction_predict(txt, i)
    if pred is None: continue
    tot+=1; hit += (pred==txt[i+1] if i+1<len(txt) else 0)
acc=hit/tot; chance=1/len(set(txt))
assert acc > 3*chance, f"induction({acc:.3f}) 应远超随机({chance:.3f})"
print(f"induction 下一字符准确率={acc:.3f} >> 随机 {chance:.3f} ✓")


### 📖 参考答案

In [ ]:
def induction_predict(text, i):
    c=text[i]
    j=text.rfind(c, 0, i)
    if j==-1 or j+1>=i: return None
    return text[j+1]
print("✓ induction head 是 in-context learning 的机制基石(复制+延续)")